In [ ]:
# Install the langgraph package
# pip install langgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.8/156.8 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.9/469.9 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.6/207.6 kB 11.5 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.79
    Uninstalling langchain-core-0.3.79:
      Successfully uninstalled langchain-core-0.3.79
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.3.27 requires langchain-core<1.0.0,>=0.3.72, but you have langchain-core 1.0.3 which is incompatible.


In [ ]:
# Import required modules
from typing_extensions import TypedDict
from langgraph.graph import StateGraph

# Define the shared state structure
class FlightState(TypedDict):
    destination: str
    destination_type: str  # "domestic" or "international"
    budget: float
    flights: list
    confirmation: str

# --- Mock Task Functions ---
def search_local_api(state: FlightState):
    print("🔍 Searching local flights...")
    return {"flights": ["LocalFlight1", "LocalFlight2"]}

def search_global_api(state: FlightState):
    print("🌐 Searching global flights...")
    return {"flights": ["GlobalFlight1", "GlobalFlight2"]}

def verify_documents(state: FlightState):
    print("🛂 Verifying documents...")
    return {}

def calculate_exchange_rates(state: FlightState):
    print("💱 Calculating exchange rates...")
    return {}

def filter_by_budget(state: FlightState):
    print(f"💸 Filtering under budget ${state['budget']}")
    return {"flights": state["flights"]}

def confirm_booking(state: FlightState):
    flights = state.get("flights", [])
    confirmation = f"✅ Confirmed: {flights[0]}" if flights else "❌ No flights available"
    print(confirmation)
    return {"confirmation": confirmation}

# --- Build the HTN Graph ---
builder = StateGraph(FlightState)
builder.set_entry_point("start")

# Entry router node: chooses domestic or international
def route_by_destination(state: FlightState) -> str:
    return "domestic" if state["destination_type"] == "domestic" else "international"

builder.add_node("start", lambda state: {})
builder.add_node("search_local", search_local_api)
builder.add_node("search_global", search_global_api)
builder.add_node("verify_docs", verify_documents)
builder.add_node("calc_exchange", calculate_exchange_rates)
builder.add_node("filter_budget", filter_by_budget)
builder.add_node("confirm", confirm_booking)

# Conditional routing from 'start'
builder.add_conditional_edges(
    "start",
    route_by_destination,
    {
        "domestic": "search_local",
        "international": "search_global"
    }
)

# Domestic path: search_local → filter_budget → confirm
builder.add_edge("search_local", "filter_budget")
builder.add_edge("filter_budget", "confirm")

# International path: search_global → verify_docs → calc_exchange → filter_budget → confirm
builder.add_edge("search_global", "verify_docs")
builder.add_edge("verify_docs", "calc_exchange")
builder.add_edge("calc_exchange", "filter_budget")
# Note: filter_budget already points to "confirm"

# Set the finish node
builder.set_finish_point("confirm")

# Compile the graph
graph = builder.compile()

# --- Run Examples ---
print("\n📦 Running domestic booking:")
state_domestic = {
    "destination": "Chicago",
    "destination_type": "domestic",
    "budget": 300,
}
graph.invoke(state_domestic)

print("\n🌍 Running international booking:")
state_international = {
    "destination": "Paris",
    "destination_type": "international",
    "budget": 800,
}
graph.invoke(state_international)


📦 Running domestic booking:
🔍 Searching local flights...
💸 Filtering under budget $300
✅ Confirmed: LocalFlight1

🌍 Running international booking:
🌐 Searching global flights...
🛂 Verifying documents...
💱 Calculating exchange rates...
💸 Filtering under budget $800
✅ Confirmed: GlobalFlight1


{'destination': 'Paris',
 'destination_type': 'international',
 'budget': 800,
 'flights': ['GlobalFlight1', 'GlobalFlight2'],
 'confirmation': '✅ Confirmed: GlobalFlight1'}